# RAG Workshop - Aufgaben

## Ziel
In diesem Workshop baust du Schritt für Schritt dein eigenes RAG-System für **SEC 10-Q Finanzdokumente**. 
Am Ende kannst du Fragen über Firmen-Filings stellen und bekommst KI-generierte Antworten basierend auf echten Finanzdaten.

## Setup
Installiere zuerst alle Dependencies:
```bash
pip install -r requirements.txt
```

## Daten
Die PDFs befinden sich im `data/` Ordner:
- **20 SEC 10-Q Filings** von 5 Tech-Firmen (AAPL, AMZN, INTC, MSFT, NVDA)
- Quartale: Q3 2022, Q1-Q3 2023
- Plus 241 Test-Fragen in `questions.csv`

## Google AI Studio API Key
1. Gehe zu https://aistudio.google.com/api-keys
2. Erstelle einen API Key
3. Trage ihn unten ein

In [ ]:
# Trage hier deinen Google AI Studio API Key ein
import os
os.environ['GOOGLE_API_KEY'] = 'DEIN_API_KEY_HIER'

---

# Milestone 1: BM25 Keyword-Suche (10 Minuten)

**Ziel:** Verstehe traditionelle Keyword-basierte Suche und ihre Grenzen

**Aufgabe:**
1. Lade SEC 10-Q PDF Dokumente aus dem `data/` Ordner
   - Nutze `glob.glob("data/*.pdf")` und `PyPDF2.PdfReader`
   - Extrahiere Text aus allen Seiten
   - Extrahiere Firmenname aus Dateiname (z.B. "2023 Q3 AAPL.pdf" → "AAPL")
2. Implementiere BM25-Suche mit der `rank_bm25` Bibliothek
3. Teste diese Query: `"What are NVIDIA's main revenue sources?"`
4. Finde einen Fall, wo BM25 **versagt** (z.B. bei Synonymen)

**Hinweis:** Nutze `BM25Okapi` aus `rank_bm25`. Tokenisiere mit `.lower().split()`. Evaluiere deine Ergebnisse mit `.get_scores()`. 
**Erwartung:** Für die NVIDIA-Query sollten NVIDIA-Dokumente (NVDA) oben im Ranking sein!

In [ ]:
# Dein Code hier



---

# Milestone 2: Embedding-basierte Suche (10 Minuten)

**Ziel:** Semantische Suche mit Embeddings

**Aufgabe:**
1. Verwende `SentenceTransformer` mit dem Modell `'all-MiniLM-L6-v2'`
2. Erstelle Embeddings für die gleichen Dokumente
3. Teste die **gleiche Query** wie bei BM25: `"What are NVIDIA's main revenue sources?"`
4. Vergleiche: **Welche Firmen** sind in den Top-3? (Sollten nur NVDA Dokumente sein!)
5. **Jetzt der Test**: Umformulierung mit Synonymen → `"What are NVIDIA's primary income streams?"`
   - BM25: Findet es noch NVIDIA-Dokumente?
   - Embeddings: Funktioniert es trotz anderer Wörter?

**Hinweis:** Nutze `cosine_similarity` aus `sklearn.metrics.pairwise`. Vergleiche die **Rankings** (welche Firmen auf Platz 1-3), nicht die numerischen Scores!

In [ ]:
# Dein Code hier



---

# Milestone 3: ChromaDB Vector Database mit Metadaten (15 Minuten)

**Ziel:** Nutze eine echte Vector Database mit strukturierten Metadaten

**Aufgabe:**
1. Erstelle einen ChromaDB Client (in-memory)
2. Erstelle eine Collection mit dem `SentenceTransformerEmbeddingFunction`
3. Extrahiere Metadaten aus den Dateinamen:
   - `company`: Der Firmenname (z.B. "AAPL" aus "2023 Q3 AAPL.pdf")
   - `quarter`: Das Quartal (z.B. "2023 Q3" aus "2023 Q3 AAPL.pdf")
4. Füge alle Dokumente **mit Metadaten** in ChromaDB ein
5. Teste normale Suche: `"What are the main business risks?"`
6. Teste Metadata-Filtering: Suche nur in Apple-Dokumenten mit `where={"company": "AAPL"}`

**Hinweis:** Verwende `chromadb.Client()` und `embedding_functions` aus `chromadb.utils`. 
Die Metadaten helfen später, Suchen auf bestimmte Firmen oder Quartale einzugrenzen!

In [ ]:
# Dein Code hier



---

# Milestone 4: RAG Pipeline mit Google Gemini (15 Minuten)

**Ziel:** Baue die komplette RAG-Pipeline

**Aufgabe:**
1. Schreibe eine Funktion `rag_query(question, collection)` die:
   - ChromaDB nach relevanten Dokumenten durchsucht
   - Die Top-3 Dokumente als Context formatiert
   - Einen Prompt erstellt: "Context: ...\n\nQuestion: ...\n\nAnswer:"
   - Google Gemini aufruft und die Antwort zurückgibt
2. Teste mit: "What are Apple's main revenue sources in Q3 2022?"
3. Vergleiche: Frage Gemini **ohne** RAG die gleiche Frage - was ist der Unterschied?

**Hinweis:** Nutze `from google import genai` und `genai.Client()` mit Modell `gemini-3.5-flash`

In [ ]:
# Dein Code hier



---

# Milestone 5: Hybrid Search (8 Minuten)

**Ziel:** Kombiniere BM25 und Embeddings für bessere Ergebnisse

**Aufgabe:**
1. Implementiere eine `hybrid_search()` Funktion
2. Hole Top-10 Ergebnisse von BM25
3. Hole Top-10 Ergebnisse von ChromaDB
4. Kombiniere beide mit Reciprocal Rank Fusion (RRF)
5. Teste mit der Query: `"operating expenses and costs"`
6. Vergleiche die Ergebnisse: **Welche Firmen** sind in BM25 Top-3, ChromaDB Top-3, und Hybrid Top-3?

**Hinweis:** RRF Formel: `score = 1/(k + rank)` für beide Listen, dann summieren. 
Hybrid ist "besser", wenn es die relevantesten Dokumente aus beiden Methoden kombiniert!

In [ ]:
# Dein Code hier



---

# Milestone 6: Reranking (7 Minuten)

**Ziel:** Verbessere Retrieval-Qualität mit Reranking

**Aufgabe:**
1. Hole Top-10 Dokumente aus ChromaDB
2. Nutze ein Cross-Encoder Modell (`cross-encoder/ms-marco-MiniLM-L-6-v2`) für Reranking
3. Berechne Relevanz-Scores zwischen Query und jedem Dokument
4. Sortiere neu und nimm die Top-3
5. Teste mit der Query: `"What are the company's liquidity and capital resources?"`
6. Vergleiche: Sind die rerankten Ergebnisse besser?

**Hinweis:** `from sentence_transformers import CrossEncoder`

In [ ]:
# Dein Code hier



---

# Milestone 7: Evaluiere dein RAG-System (10 Minuten)

**Ziel:** Miss die Performance deines Systems mit echten Test-Fragen

**Aufgabe:**
1. Lade die Test-Fragen aus `data/questions.csv`
2. Starte mit **Single-Doc Single-Chunk** Fragen (einfach)
3. Berechne für jede Frage:
   - **MRR (Mean Reciprocal Rank)**: Position des ersten relevanten Dokuments (1/rank)
   - Rank 1 → MRR = 1.0, Rank 2 → MRR = 0.5, Rank 3 → MRR = 0.33
4. **Bonus:** Wenn Zeit bleibt, versuche **Multi-Doc RAG** Fragen (schwieriger - vergleicht mehrere Quartale)

**Hinweis:** Die Funktion `evaluate_rag_system()` ist vorbereitet - du musst sie nur aufrufen. MRR ist besser als Precision@3 für Single-Doc Aufgaben, da es die Position bewertet!

In [ ]:
import pandas as pd

# Lade Test-Fragen aus CSV
questions_df = pd.read_csv("data/questions.csv")

# Zeige verfügbare Schwierigkeitsgrade
print("📊 Verfügbare Question Types:")
print(questions_df['Question Type'].value_counts())
print()

def evaluate_rag_system(rag_function, collection, difficulty="Single-Doc Single-Chunk RAG", num_questions=5):
    """
    Evaluiert dein RAG-System mit echten Test-Fragen
    
    Args:
        rag_function: Deine rag_query() Funktion
        collection: Deine ChromaDB Collection
        difficulty: "Single-Doc Single-Chunk RAG" (einfach) oder "Multi-Doc RAG" (schwer)
        num_questions: Anzahl Test-Fragen
    """
    # Filtere nach Schwierigkeitsgrad
    test_questions = questions_df[questions_df['Question Type'] == difficulty].head(num_questions)
    
    if len(test_questions) == 0:
        print(f"❌ Keine Fragen für Difficulty '{difficulty}' gefunden!")
        return 0
    
    total_mrr = 0
    
    print(f"🔍 Evaluiere RAG-System ({difficulty})...\n")
    print("="*80)
    
    for idx, row in test_questions.iterrows():
        question = row['Question']
        source_docs = row['Source Docs']  # z.B. "*AAPL*" oder "*2023 Q3 AAPL*"
        
        # Hole Top-3 Dokumente
        results = collection.query(query_texts=[question], n_results=3)
        retrieved_docs = results['metadatas'][0] if results['metadatas'] else []
        
        # Entferne Sterne für Vergleich
        expected_doc = source_docs.replace('*', '')
        
        # Prüfe ob Multi-Doc (nur Firmenname) oder Single-Doc (mit Quartal)
        # Multi-Doc: "AAPL" (nur 4 Zeichen)
        # Single-Doc: "2023 Q3 AAPL" (länger)
        is_multi_doc = len(expected_doc.split()) == 1
        
        # Berechne MRR (Mean Reciprocal Rank)
        # Finde die Position des ersten relevanten Dokuments
        mrr = 0
        for rank, doc_meta in enumerate(retrieved_docs, start=1):
            if is_multi_doc:
                # Multi-Doc: Nur Firma muss matchen
                expected_company = expected_doc
                if 'company' in doc_meta and doc_meta['company'] == expected_company:
                    mrr = 1.0 / rank
                    break
            else:
                # Single-Doc: Exaktes Dokument (Firma UND Quartal)
                if 'title' in doc_meta and doc_meta['title'] == expected_doc:
                    mrr = 1.0 / rank
                    break
        
        total_mrr += mrr
        
        rank_str = f"Rank {int(1/mrr)}" if mrr > 0 else "Not Found"
        print(f"❓ {question[:80]}...")
        print(f"   Expected: {expected_doc} | MRR: {mrr:.3f} ({rank_str})")
        print()
    
    avg_mrr = total_mrr / len(test_questions)
    
    print("="*80)
    print(f"🎯 DEIN SCORE (MRR): {avg_mrr:.3f}/1.000")
    print("="*80)
    
    # Leaderboard (angepasste Schwellenwerte für MRR)
    if avg_mrr >= 0.9:
        print("🏆 EXCELLENT! Du bist ein RAG-Meister!")
    elif avg_mrr >= 0.7:
        print("👍 GUT! Dein System funktioniert solide.")
    elif avg_mrr >= 0.5:
        print("📈 OK! Es gibt noch Verbesserungspotential.")
    else:
        print("🔧 Versuch Hybrid Search oder Reranking!")
    
    return avg_mrr

# Führe die Evaluation aus (entferne das # am Anfang):
# score = evaluate_rag_system(rag_query, collection, difficulty="Single-Doc Single-Chunk RAG", num_questions=5)

# Bonus: Wenn Zeit noch nicht rum ist, versuche die schwierigeren Multi-Doc Fragen!
# score_hard = evaluate_rag_system(rag_query, collection, difficulty="Multi-Doc RAG", num_questions=5)

---

## 🎉 Glückwunsch!

Du hast erfolgreich ein RAG-System für Finanzdokumente gebaut! 

### Was du gelernt hast:
- ✅ BM25 Keyword-Suche vs. Semantic Search
- ✅ PDF-Verarbeitung und Text-Extraktion
- ✅ Embeddings und Vector Databases (ChromaDB)
- ✅ RAG Pipeline mit LLM (Google Gemini 3.5 Flash)
- ✅ Metadata-Filtering (nach Firma, Quartal)
- ✅ Hybrid Search & Reranking
- ✅ Evaluation von Retrieval-Qualität

### Use Cases:
- 📊 Finanz-Analyse: "Was sind die Hauptrisiken von Tesla?"
- 💰 Investment-Research: "Wie haben sich Apples Einnahmen entwickelt?"
- 📈 Wettbewerbs-Analyse: "Vergleiche die R&D-Ausgaben von Tech-Firmen"

### Nächste Schritte:
- Teste mit eigenen PDF-Dokumenten
- Experimentiere mit Chunking-Strategien (500-1000 Token Chunks)
- Probiere persistent Storage: `chromadb.PersistentClient(path="./chroma_db")`
- Verbessere die Prompts für bessere Antworten